# QQQ Top-20 → Max-Sharpe Portfolio, Rebalanced Monthly

Back-test of the following strategy against two buy-and-hold baselines.

| | |
|---|---|
| **Principal** | $10,000, invested once at the start |
| **Universe** | Nasdaq-100 constituents (what QQQ holds) |
| **Selection** | The dashboard's own screen — `screening.evaluate_ticker` — run point-in-time, ranked by `daily_annret`, top 20 |
| **Sizing** | Long-only maximum-Sharpe mean-variance weights over the selected names |
| **Rebalance** | Monthly: new selection and new weights at the first trading day of each month, held to the first trading day of the next |
| **Window** | 2025-06-01 → today |
| **Baselines** | Buy and hold QQQ; buy and hold BOXX |

The engine lives in `qqq_maxsharpe.py`, one directory up. It is a module rather than
cells in this file so that `qqq_backtest_visualization.ipynb` draws the same strategy
this one measures, instead of a second copy that drifts.

## What "the same stock selection" means here

`screening.py` is the Screener tab's engine. `evaluate_ticker` applies, in order:

1. a price gate (`min_price`, $10) and a liquidity gate (`min_avg_volume`, 200k on the 50-day average),
2. Minervini Stage-2: close above both the 150- and 200-day SMA, and both of those SMAs rising over the last 20 bars,
3. within 25% of the 52-week high,
4. positive 6-month (126-bar) relative strength versus `^GSPC`.

`run_screening` then sorts everything that passes by `daily_annret` — the annualised
return over the trailing window. `qqq_maxsharpe.screen_asof` calls **the same
function**, not a re-implementation, so the selection cannot silently diverge from the
dashboard's.

The one change is *when* it is asked. `evaluate_ticker` reads `.iloc[-1]`: it always
answers "as of the last bar you hand it". The dashboard hands it today's history; the
module hands it history truncated at each rebalance date, which makes the same code
point-in-time. Nothing downstream of the truncation can see the future.

## Known limitations — read these before trusting a number

* **Survivorship / membership bias.** The universe is *today's* Nasdaq-100, held
  fixed across the whole window. Names added during 2025-26 are in the pool before
  they were actually in QQQ, and names dropped are missing. Over a 15-month window
  this is small but it is not zero, and it biases the strategy's return upward.
* **Fractional shares** are assumed. With $10,000 across 20 names the average
  position is $500, and several Nasdaq-100 names trade above that, so whole-share
  rounding would distort the weights more than fractional fills distort realism.
* **No taxes.** Monthly rebalancing in a taxable account realises short-term gains.
* **Costs** are the repo's `backtest.CostModel` default — 5 bps slippage, 1 bp
  commission, one way — applied to every fill.
* **Fifteen months is not a sample.** Everything below is one path through one
  regime. It cannot distinguish skill from a good quarter for large-cap tech.

In [ ]:
from __future__ import annotations

import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

# The notebook lives in notebooks/; the strategy modules live one level up.
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import qqq_maxsharpe as qms
from backtest import CostModel

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

print(f"repo root: {REPO_ROOT}")

## 1 · Configuration

`BacktestSettings` is the whole knob panel. `SCREENING_PARAMS` is imported from
`config.py` untouched, so changing the screen in the dashboard changes it here too.

In [ ]:
CFG = qms.BacktestSettings()
START, END = CFG.start_ts, CFG.end_ts
CFG

## 2 · Universe — what QQQ holds

Scraped from Wikipedia the same way `screening.load_sp500_symbols` scrapes the S&P 500.
The static list is the fallback for an offline or rate-limited run; it is a snapshot,
which is where the membership bias flagged at the top comes from.

In [ ]:
UNIVERSE = qms.nasdaq100_symbols()
len(UNIVERSE), UNIVERSE[:12]

## 3 · Price history

`data.load_history` is the repo's loader: on-disk CSV cache first, then yfinance.
It returns an empty frame rather than raising when a symbol is unavailable, and it pulls
with `auto_adjust=True` — so QQQ's dividends are already in its price series and the
comparison is total return against total return, not price against price.

**The cache does not expire.** `load_history` reuses `.screen_cache/bt_<SYMBOL>_5y.csv`
whenever it exists, so a second run weeks later replays the old history. Delete those
files to pull fresh data:

```bash
rm .screen_cache/bt_*_5y.csv
```

The first live run fetches ~100 symbols one at a time and takes a few minutes.

If fewer than half the universe comes back — no network, an egress policy that blocks
Yahoo, a throttled session — the module switches to `data.synthetic_ohlcv` so the
machinery still runs end to end. **A synthetic run proves the code works and says
nothing whatsoever about the strategy.** The banner below tells you which mode you are in.

In [ ]:
MARKET = qms.load_market(UNIVERSE, CFG)
CLOSES, OPENS, CALENDAR = MARKET.closes, MARKET.opens, MARKET.calendar
SYNTHETIC = MARKET.synthetic

## 4 · Selection — the dashboard's screen, asked point-in-time

`screen_asof` truncates each symbol's history at the as-of date and hands it to
`screening.evaluate_ticker`, so `.iloc[-1]` inside that function means "the last bar
that had printed by then".

One shim, applied in the module: the screen resolves each name's sector through
yfinance's `.info` endpoint. Sector is a *label* on the result — `passed` and
`daily_annret`, the only two fields the ranking reads, never touch it — so it is
stubbed out rather than firing one network call per name per rebalance.

In [ ]:
# Smoke test on the first as-of date in the window.
_probe_date = CLOSES.index[CLOSES.index < START][-1]
_probe = qms.screen_asof(MARKET, _probe_date, UNIVERSE, CFG)
print(f"as of {_probe_date.date()}: {len(_probe)} of {len(UNIVERSE)} names passed the screen")
_probe.head(20)[["ticker", "last_close", "daily_annret", "ann_vol", "rs6m_vs_mkt",
                 "within_52w_high_pct"]] if not _probe.empty else "none passed"

## 5 · Sizing — long-only maximum Sharpe

Over the selected names, `qms.max_sharpe_weights` maximises

$$\text{Sharpe}(w) = \frac{w^{\top}\mu - r_f}{\sqrt{w^{\top}\Sigma w}}
\qquad \text{s.t.}\quad \sum_i w_i = 1,\quad 0 \le w_i \le w_{\max}$$

on annualised trailing estimates. Three things keep it from producing garbage:

* **Shrinkage.** A 252×20 sample covariance is badly conditioned; the raw inverse
  concentrates the whole book in whichever name happened to be quietest. `cov_shrinkage`
  pulls the off-diagonals toward zero.
* **A weight cap.** `max_weight` = 25%, mirroring `config.MAX_POSITION_PCT`.
* **A min-variance fallback.** When no selected name has a trailing excess return above
  $r_f$, the Sharpe objective has no meaningful maximum. Rather than return whatever the
  optimiser lands on, it falls back to minimum variance under the same constraints and
  says so in the plan's `note` column.

The optimiser is multi-start (equal weight, inverse volatility, and random draws) because
SLSQP on this objective is not convex and will happily stop at a local optimum.

$r_f$ is the trailing annualised return of `rf_proxy` (BOXX), so the hurdle the optimiser
prices against is the same instrument the back-test compares against.

In [ ]:
# What the optimiser does with the first month's picks.
_picks = qms.select_top_n(MARKET, _probe_date, UNIVERSE, CFG)
_rets = qms.selected_returns(MARKET, _probe_date, _picks, CFG)
_rf = qms.risk_free_rate(MARKET, _probe_date, CFG)
_w, _note = qms.max_sharpe_weights(_rets, _rf, CFG)

_mu, _cov = qms.annualised_moments(_rets, CFG.cov_shrinkage)
print(f"rf (from {CFG.rf_proxy}): {_rf:.2%} | solver: {_note}")
print(f"portfolio: return {qms.portfolio_return(_w, _mu):.2%}  "
      f"vol {qms.portfolio_vol(_w, _cov):.2%}  "
      f"Sharpe {(qms.portfolio_return(_w, _mu) - _rf) / qms.portfolio_vol(_w, _cov):.2f}")
_w[_w > 0.005].sort_values(ascending=False).map(lambda v: f"{v:.1%}").to_frame("weight")

## 6 · The back-test

The month boundary is the only place anything trades:

* **Signal date** — the last close *before* the first trading day of the month. The screen
  and the covariance both stop there.
* **Execution** — the open of the first trading day of the month, at
  `CostModel.fill_price`, with commission on the notional.
* **Rebalance** — the roll from last month's weights to this month's happens in that one
  trade: names that dropped out are sold, survivors are adjusted to their new weight.
  Holding to the end of the month and re-entering is the same thing with a gap in
  exposure, so the position is carried straight across the boundary.
* **Marking** — equity is marked at the close every day in between; nothing trades.

Nothing in the loop reads a bar later than the signal date when choosing what to hold,
which is the property the repo's `tests/test_causality.py` asserts for the breakout engine.

In [ ]:
EQUITY, TRADES, PLAN = qms.run_strategy(MARKET, UNIVERSE, CFG)
CURVES = qms.curves(MARKET, EQUITY, CFG)

print(f"rebalances: {len(PLAN)} | fills: {len(TRADES)} | "
      f"commission: ${TRADES['cost'].sum():,.2f} (slippage is inside every fill price) | "
      f"lowest cash balance: ${EQUITY['cash'].min():,.2f}")
CURVES.tail()

## 7 · Results

`qms.equity_metrics` mirrors `backtest._compute_metrics` — same Sharpe (rf = 0, on daily
equity returns), same CAGR, same drawdown definition — so a figure here is comparable to
one printed by `backtest.py`. Every curve starts at the principal, so **Profit** is profit
on the $10,000, with day one's move and the cost of getting in already inside it.

In [ ]:
SUMMARY = pd.DataFrame({name: qms.equity_metrics(CURVES[name]) for name in CURVES.columns}).T
PCT = ["Total return", "CAGR", "Ann. vol", "Max drawdown", "Best day", "Worst day"]

display_summary = SUMMARY.copy()
for col in PCT:
    display_summary[col] = display_summary[col].map(lambda v: f"{v:.2%}")
for col in ["Ending equity", "Profit"]:
    display_summary[col] = display_summary[col].map(lambda v: f"${v:,.0f}")
for col in ["Sharpe (rf=0)", "Calmar"]:
    display_summary[col] = display_summary[col].map(lambda v: f"{v:.2f}")

print(f"${CFG.principal:,.0f} from {CALENDAR[0].date()} to {CALENDAR[-1].date()}"
      + ("   [SYNTHETIC DATA]" if SYNTHETIC else ""))
display_summary

In [ ]:
best = SUMMARY["Ending equity"].idxmax()
strat = SUMMARY.loc["Strategy"]
lines = [f"Winner: {best}  (${SUMMARY.loc[best, 'Ending equity']:,.0f})", ""]
for name in CURVES.columns:
    row = SUMMARY.loc[name]
    lines.append(f"  {name:<10} ${row['Ending equity']:>10,.0f}   "
                 f"profit ${row['Profit']:>9,.0f}   {row['Total return']:>8.2%}   "
                 f"Sharpe {row['Sharpe (rf=0)']:>5.2f}   maxDD {row['Max drawdown']:>7.2%}")
lines.append("")
for name in CURVES.columns:
    if name == "Strategy":
        continue
    diff = strat["Profit"] - SUMMARY.loc[name, "Profit"]
    verb = "ahead of" if diff >= 0 else "behind"
    lines.append(f"  Strategy is ${abs(diff):,.0f} {verb} buy-and-hold {name}.")
print("\n".join(lines))

### Equity curves

One axis, one currency, three series — the comparison the question actually asks for.
Each line is labelled at its right-hand end as well as in the legend, so the series are
never identified by colour alone.

For interactive versions of these — hover, a frontier plot, a weight slider — open
`qqq_backtest_visualization.ipynb`.

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

# Categorical slots 1-3 of the validated default palette (blue / orange / aqua).
SERIES_COLORS = {"Strategy": "#2a78d6", "QQQ": "#eb6834", "BOXX": "#1baf7a"}
INK, INK_MUTED, GRID = "#0b0b0b", "#52514e", "#e4e3df"
USD = FuncFormatter(lambda v, _: f"${v:,.0f}")


def end_labels(ax, entries, min_gap=0.07):
    """Right-edge series labels, nudged apart when two lines finish close together.

    `entries` is [(text, x, y, color)]. A label pushed off its own line gets a thin
    leader in the series colour, so the pairing stays unambiguous.
    """
    lo, hi = ax.get_ylim()
    gap = min_gap * (hi - lo)
    ordered = sorted(entries, key=lambda e: e[2])
    placed, last = [], None
    for _, _, y, _ in ordered:
        last = y if last is None else max(y, last + gap)
        placed.append(last)
    for target, (text, x, y, color) in zip(placed, ordered):
        ax.annotate(text, (x, target), color=INK, fontsize=9, va="center", ha="left",
                    xytext=(8, 0), textcoords="offset points", annotation_clip=False)
        if abs(target - y) > gap * 0.2:
            ax.plot([x, x], [y, target], color=color, linewidth=0.8, alpha=0.55, clip_on=False)


def style_axes(ax, title, ylabel, formatter=USD):
    ax.set_title(title, color=INK, fontsize=13, pad=12, loc="left")
    ax.set_ylabel(ylabel, color=INK_MUTED, fontsize=10)
    ax.yaxis.set_major_formatter(formatter)
    ax.grid(axis="y", color=GRID, linewidth=0.8)
    ax.set_axisbelow(True)
    for side in ("top", "right", "left"):
        ax.spines[side].set_visible(False)
    ax.spines["bottom"].set_color(GRID)
    ax.tick_params(colors=INK_MUTED, labelsize=9, length=0)


fig, ax = plt.subplots(figsize=(11, 5.5), dpi=130)
labels = []
for name in CURVES.columns:
    series = CURVES[name].dropna()
    color = SERIES_COLORS.get(name, "#eda100")
    ax.plot(series.index, series.values, color=color, linewidth=2.0, label=name,
            solid_capstyle="round")
    labels.append((f"{name} ${series.iloc[-1]:,.0f}", series.index[-1], series.iloc[-1], color))

ax.axhline(CFG.principal, color=INK_MUTED, linewidth=1.0, linestyle=(0, (4, 4)), zorder=0)
style_axes(ax, f"${CFG.principal:,.0f} invested {CALENDAR[0].date()} — "
               + ("SYNTHETIC DATA" if SYNTHETIC else "portfolio value"), "Portfolio value")
ax.legend(frameon=False, loc="upper left", fontsize=9, labelcolor=INK_MUTED)
ax.margins(x=0.10)
end_labels(ax, labels)
fig.tight_layout()
plt.show()

### Drawdown

Peak-to-trough on the same three curves. This is the half of the comparison a total-return
number hides: BOXX is here to show what almost no drawdown looks like next to the other two.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.2), dpi=130)
for name in CURVES.columns:
    series = CURVES[name].dropna()
    dd = (series / series.cummax() - 1.0) * 100
    color = SERIES_COLORS.get(name, "#eda100")
    ax.plot(dd.index, dd.values, color=color, linewidth=2.0, label=name)
    # Label the trough rather than the right edge: it is the number the chart is
    # about, and three curves that all finish near 0% would stack their labels.
    trough = dd.idxmin()
    ax.plot([trough], [dd.min()], marker="o", markersize=6, color=color,
            markeredgecolor="#fcfcfb", markeredgewidth=1.5, zorder=5)
    ax.annotate(f"{name} {dd.min():.1f}%", (trough, dd.min()), color=INK, fontsize=9,
                va="top", ha="center", xytext=(0, -8), textcoords="offset points")

style_axes(ax, "Drawdown from running peak", "Drawdown",
           FuncFormatter(lambda v, _: f"{v:.0f}%"))
ax.axhline(0, color=GRID, linewidth=1.0)
ax.legend(frameon=False, loc="lower left", fontsize=9, labelcolor=INK_MUTED)
ax.margins(x=0.04, y=0.14)
fig.tight_layout()
plt.show()

### Month by month

Where the total came from. Grouped bars, one group per calendar month, with a 2px surface
gap between adjacent fills so neighbouring bars stay distinct without a border colour.

In [ ]:
# CURVES.iloc[0] is the principal on the day before trading starts, so the first
# month carries the cost of getting in.
monthly = pd.concat([CURVES.iloc[[0]], CURVES.resample("ME").last()]).pct_change().dropna(how="all") * 100
monthly.index = monthly.index.strftime("%Y-%m")

fig, ax = plt.subplots(figsize=(11, 4.6), dpi=130)
names = list(CURVES.columns)
width = 0.8 / len(names)
x = np.arange(len(monthly))
for i, name in enumerate(names):
    ax.bar(x + i * width - 0.4 + width / 2, monthly[name].values, width * 0.94,
           color=SERIES_COLORS.get(name, "#eda100"), label=name,
           edgecolor="#fcfcfb", linewidth=1.0)

style_axes(ax, "Monthly return", "Return", FuncFormatter(lambda v, _: f"{v:.0f}%"))
ax.axhline(0, color=INK_MUTED, linewidth=1.0)
ax.set_xticks(x)
ax.set_xticklabels(monthly.index, rotation=45, ha="right")
ax.legend(frameon=False, loc="best", fontsize=9, labelcolor=INK_MUTED)
fig.tight_layout()
plt.show()

monthly.round(2)

### What it held, and why

`PLAN` is the audit trail: one row per rebalance, with the signal date, how many names
cleared the screen, and how the weights were reached. A `min-variance fallback` note means
the screen's survivors had no trailing excess return over the risk-free proxy that month.

In [ ]:
plan_view = PLAN[["rebalance", "signal_date", "n_passed", "n_held", "note"]].copy()
plan_view["rebalance"] = plan_view["rebalance"].dt.date
plan_view["signal_date"] = plan_view["signal_date"].dt.date
plan_view["top holdings"] = [
    ", ".join(f"{s} {w:.0%}" for s, w in sorted(d.items(), key=lambda kv: -kv[1])[:6]
              if w >= 0.005) or "—"
    for d in PLAN["weights"]
]
plan_view

In [ ]:
# Full weight matrix: rebalance date x symbol. Blank = not held that month.
weight_matrix = pd.DataFrame(list(PLAN["weights"]), index=PLAN["rebalance"].dt.date)
weight_matrix = weight_matrix.reindex(sorted(weight_matrix.columns), axis=1)
print(f"{weight_matrix.shape[1]} distinct names held across {len(weight_matrix)} rebalances")
(weight_matrix * 100).round(1).replace(0.0, np.nan).fillna("")

In [ ]:
# Every fill, if you want to check the cost model or reconcile a month.
trades_view = TRADES.copy()
if not trades_view.empty:
    trades_view["date"] = trades_view["date"].dt.date
    trades_view = trades_view.round({"shares": 3, "fill": 2, "notional": 2, "cost": 2})
print(f"{len(trades_view)} fills, ${TRADES['cost'].sum():,.2f} in commission, "
      f"{TRADES['notional'].abs().sum() / CFG.principal:.1f}x principal turned over")
trades_view.head(40)

## 8 · Reading this honestly

* **The window is one regime.** June 2025 onward is fifteen months. A Sharpe computed on
  it has an enormous standard error; the ranking between the strategy and QQQ could
  plausibly invert on a different fifteen months. Treat the table as a description of what
  happened, not an estimate of what will.
* **Max-Sharpe weights are the fragile part.** Mean-variance optimisation is famously
  sensitive to the mean estimate, and a trailing 252-day mean is a weak one. The shrinkage
  and the 25% cap are there to blunt that, and they are also the reason the result is not a
  pure statement about the objective.
* **The screen already selects on momentum**, and `daily_annret` ranks on it again. Feeding
  those trailing returns into the optimiser as $\mu$ compounds the same bet three times.
  Setting `rf_proxy=None` and `cov_shrinkage=1.0` collapses the sizing toward
  risk-parity-ish weights, which is a useful contrast run.
* **BOXX is the right kind of baseline** — it is the "did taking any equity risk at all pay"
  question. QQQ is the "did the selection beat just owning the index" question. They answer
  different things and the strategy can lose to one and beat the other.

### Things worth changing and re-running

```python
CFG = qms.BacktestSettings(top_n=10)                       # concentration
CFG = qms.BacktestSettings(max_weight=1.0)                 # uncapped max-Sharpe
CFG = qms.BacktestSettings(cov_shrinkage=0.0)              # raw sample covariance
CFG = qms.BacktestSettings(start="2023-01-01")             # a longer, still short, window
CFG = qms.BacktestSettings(costs=CostModel(slippage_bps=20, commission_bps=5))
```

Re-run from the configuration cell down after changing it.